In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import pandas as pd
from sklearn import svm
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

def get_svm_data(dataset_name):
    # Transformation (normalization for SVM input)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Load both training and test sets to form a single pool
    train_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform) if dataset_name == "MNIST" else \
                 torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform) if dataset_name == "MNIST" else \
                torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    X_train = train_ds.data.reshape(-1, 784).float().numpy() / 255.0
    y_train = train_ds.targets.numpy()

    X_test = test_ds.data.reshape(-1, 784).float().numpy() / 255.0
    y_test = test_ds.targets.numpy()

    return X_train, y_train, X_test, y_test

In [ ]:
datasets = ["MNIST", "FashionMNIST"]
kernels = ['poly', 'rbf']
svm_results = []

pbar = tqdm(total=len(datasets) * len(kernels), desc="SVM Experiments")

for ds_name in datasets:
    X_train, y_train, X_test, y_test = get_svm_data(ds_name)

    for k in kernels:
        pbar.set_description(f"Processing {ds_name} | Kernel: {k}")
        start_time = time.time()

        # verbose=True will print progress to the console
        clf = svm.SVC(kernel=k, verbose=True)

        try:
            # Training with all samples
            clf.fit(X_train, y_train)
            duration_ms = (time.time() - start_time) * 1000

            preds = clf.predict(X_test)
            acc = accuracy_score(y_test, preds) * 100


            svm_results.append({
                "Dataset": ds_name,
                "Kernel": k,
                "Test Accuracy (%)": round(acc, 2),
                "Train Time (ms)": round(duration_ms, 2)
            })
        except Exception as e:
            print(f"Crash or Error occurred for {ds_name} with {k} kernel: {e}")

        pbar.update(1)

pbar.close()

SVM Experiments:   0%|          | 0/4 [00:00<?, ?it/s]

[LibSVM][LibSVM][LibSVM][LibSVM]

In [ ]:
# Convert to DataFrame and save to CSV
df_svm = pd.DataFrame(svm_results)
df_svm.to_csv("q1b_svm_results.csv", index=False)

# Display for report entry
print("\nFinal Results for Q1(b)")
print(df_svm)


Final Results for Q1(b)
        Dataset Kernel  Test Accuracy (%)  Train Time (ms)
0         MNIST   poly              97.71        288743.64
1         MNIST    rbf              97.92        268346.94
2  FashionMNIST   poly              86.30        473309.68
3  FashionMNIST    rbf              88.29        389151.25
